In [ ]:
# Uninstall conflicting packages
!pip uninstall -y transformers torchvision torch

# Install compatible versions
!pip install torch==2.1.0 torchvision==0.16.0
!pip install transformers==4.33.1

# Then install Surya
!pip install git+https://github.com/VikParuchuri/surya.git
!pip install surya-ocr



In [ ]:
!surya_ocr --help

## Restart after this cell

In [2]:
!pip install pdfplumber pdf2image
!apt-get update
!apt-get install -y poppler-utils

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 70.9 MB/s eta 0:00:00:00:0100:01
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,086 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,389 kB]  
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jamm

## Import Libraries

In [3]:
import os
import os
import re
import json
import subprocess
from io import StringIO
from pathlib import Path
from pdf2image import convert_from_path
from pdfminer.high_level import extract_text_to_fp
from pdfminer.pdfpage import PDFPage
import shutil
import pdfplumber
from PIL import Image


## Configuration

In [4]:
INPUT_DIR = Path("/kaggle/input/criminal/criminal")
OUTPUT_DIR = Path("/kaggle/working/criminal/criminal_Extracted")
OCR_OUTPUT_DIR = Path("/kaggle/working/ocr_out")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OCR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Functions

In [5]:
def contains_garbled_text(text: str, max_consonant_seq=5, quote_bracket_seq=4):
    """
    Detects garbled text including multi-line quotes/brackets:
    - Unicode or legacy Sinhala characters
    - Multiple semicolons or commas inside a word
    - Corrupt symbols like <>{}^+@|
    - Long consonant sequences
    """
    unicode_sinhala_pattern = re.compile(r"[\u0D80-\u0DFF]")
    legacy_sinhala_pattern = re.compile(r"[úïñÿ]")
    corrupt_symbols_pattern = re.compile(r"[<>{}^+@\\|]")
    bracket_quote_pattern = re.compile(r'[\(\["“‘](.+?)[\)\]”’"]', re.DOTALL)

    if unicode_sinhala_pattern.search(text) or legacy_sinhala_pattern.search(text):
        return True

    lines = text.splitlines()
    for line in lines:
        words = line.split()
        for word in words:
            if word.count(";") > 1 or word.count(",") > 1:
                return True
            if corrupt_symbols_pattern.search(word):
                return True

    # Check multi-line quoted/bracketed sequences
    for match in bracket_quote_pattern.findall(text):
        clean_match = re.sub(r"^[^\w]+|[^\w]+$", "", match)
        if not clean_match:
            continue
        # long consonant sequences
        if re.search(
            r"[bcdfghjklmnpqrstvwxzBCDFGHJKLMNPQRSTVWXZ;=]{%d,}" % quote_bracket_seq,
            clean_match,
        ):
            return True

    # normal consonant sequences
    for line in lines:
        for word in line.split():
            if not word[0].isupper() and re.search(
                r"[bcdfghjklmnpqrstvwxzBCDFGHJKLMNPQRSTVWXZ;=]{%d,}"
                % max_consonant_seq,
                word,
            ):
                return True
    return False


# === CLEAN TEXT PROCESSING ===
def clean_paragraphs(text):
    """Clean and format text using the improved method from 1_extract_all_pdfs_to_text.py"""
    # Step 1: Fix hyphenation across lines/pages
    text = re.sub(r"-\n", "", text)

    # Step 2: Join broken lines
    # Join lines that end with lowercase letters (most common case)
    text = re.sub(r"([a-z])\n([a-z])", r"\1 \2", text)

    # Join lines that end with commas
    text = re.sub(r",\n([a-z])", r", \1", text)

    # Join lines that don't end with sentence punctuation and next line starts with lowercase
    text = re.sub(r"([^.!?])\n([a-z])", r"\1 \2", text)

    # Join lines that end with words and next line starts with lowercase
    text = re.sub(r"(\w)\n([a-z])", r"\1 \2", text)

    # Step 3: Turn remaining line breaks within paragraphs into spaces
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Step 4: Normalize multiple spaces
    text = re.sub(r" {2,}", " ", text)

    # Step 5: Preserve real paragraph breaks (2+ newlines)
    text = re.sub(r"\n{2,}", "\n\n", text)

    return text.strip()

In [12]:
# === OCR PROCESSING ===
def extract_surya_text_from_page(page_output_dir: str) -> str:
    """Recursively find JSON from Surya output and extract text lines."""
    json_file = None
    for root, _, files in os.walk(page_output_dir):
        for f in files:
            if f.endswith(".json"):
                json_file = os.path.join(root, f)
                break
        if json_file:
            break

    if not json_file:
        print(f"⚠️ No JSON found in {page_output_dir}")
        return ""

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    texts = []
    for page_key, samples in data.items():
        for sample_item in samples:
            for line in sample_item.get("text_lines", []):
                line_text = line.get("text", "").strip()
                if line_text:
                    texts.append(line_text)
    return " ".join(texts)


def process_page_with_ocr(pdf_path: Path, page_num: int) -> str:
    """Process a single page with OCR when Sinhala content is detected."""
    print(f"   🔎 Using Surya OCR for page {page_num}")
    
    images = convert_from_path(str(pdf_path), first_page=page_num, last_page=page_num)
    temp_image_path = f"/tmp/page_{page_num}.jpg"
    images[0] = images[0].convert("RGB")
    images[0].save(temp_image_path, "JPEG")
    images[0].save(temp_image_path, "JPEG")

    page_output_dir = os.path.join(OCR_OUTPUT_DIR, f"{pdf_path.stem}_page_{page_num}")
    os.makedirs(page_output_dir, exist_ok=True)

    # Run Surya OCR
    cmd = f'surya_ocr "{temp_image_path}" --images --output_dir "{page_output_dir}"'
    with open(os.devnull, "w") as fnull:
        subprocess.run(cmd, shell=True,stdout=fnull,
            stderr=fnull, check=True)

    # Extract text from OCR output
    surya_text = extract_surya_text_from_page(page_output_dir)

    # Clean up temporary files
    if os.path.exists(temp_image_path):
        os.remove(temp_image_path)
    if os.path.exists(page_output_dir):
        shutil.rmtree(page_output_dir)

    return surya_text


def extract_text_from_pdf_page_by_page(pdf_path: Path) -> list:
    """Extract text from PDF page by page using pdfplumber for better text extraction."""
    page_texts = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_text = page.extract_text()
                if page_text:
                    page_texts.append(page_text)
                else:
                    page_texts.append("")
    except Exception as e:
        print(f"❌ Error reading {pdf_path.name}: {e}")
        # Fallback to pdfminer if pdfplumber fails
        try:
            with open(pdf_path, "rb") as f:
                for i, page in enumerate(PDFPage.get_pages(f)):
                    output_string = StringIO()
                    extract_text_to_fp(f, output_string, page_numbers=[i])
                    page_texts.append(output_string.getvalue())
        except Exception as e2:
            print(f"❌ Fallback also failed: {e2}")
            return [""] 

    return page_texts

In [13]:
def process_pdf(pdf_path: Path, output_dir: Path, ocr_output_dir: Path):
    """Process one PDF using the merged approach."""
    output_file = output_dir / (pdf_path.stem + ".txt")
    page_texts = extract_text_from_pdf_page_by_page(pdf_path)
    final_text_pages = []

    for i, page_text in enumerate(page_texts):
        page_num = i + 1
        print(f"➡️ {pdf_path.name} - Page {page_num}/{len(page_texts)}")

        # Decide: OCR or clean text extraction
        if contains_garbled_text(page_text) or not page_text.strip():
            # Use OCR for Sinhala content or empty pages
            ocr_text = process_page_with_ocr(pdf_path, page_num)
            cleaned = clean_paragraphs(ocr_text)
            final_text_pages.append(cleaned)
        else:
            # Use clean text extraction for non-Sinhala content
            print(f"   ✅ Using clean text extraction for page {page_num}")
            cleaned = clean_paragraphs(page_text)
            final_text_pages.append(cleaned)

    with open(output_file, "w", encoding="utf-8") as f:
        for line in final_text_pages:
            f.write(line + "\n\n")

    print(f"   📄 Saved {output_file}")


def process_all_pdfs(input_dir: Path, output_dir: Path, ocr_output_dir: Path):
    """Process all PDF files in the input directory."""
    pdf_files = list(input_dir.rglob("*.pdf"))
    print(f"📂 Found {len(pdf_files)} PDF files...")

    for pdf_file in pdf_files:
        relative_path = pdf_file.relative_to(input_dir)
        output_file = output_dir / relative_path.with_suffix(".txt")
        output_file.parent.mkdir(parents=True, exist_ok=True)
        
        if output_file.exists():
            print(f"➡️ Skipping {pdf_file.name} (already processed)")
            continue
        process_pdf(pdf_file, output_dir, ocr_output_dir)

    print("🎉 All PDFs processed.")

## Inference

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # suppress all warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # hides TensorFlow/XLA logs
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/local/cuda'  # optional

process_all_pdfs(INPUT_DIR, OUTPUT_DIR, OCR_OUTPUT_DIR)